<a href="https://colab.research.google.com/github/Ugradcoder-ys/mangan-ji-pepper-counting/blob/main/12_14_rf_detr%2BBoostTrack.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#環境構築

In [ ]:
# 必要なライブラリのインストール
# ultralytics: YOLOv11用
# deep-sort-realtime: DeepSORTの実装ライブラリ
# roboflow: データセットダウンロード用 (追加)
!pip install ultralytics deep-sort-realtime roboflow

# Google Driveのマウント（データセットや動画の保存先）
from google.colab import drive
drive.mount('/content/drive')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 28.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 59.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 38.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 77.0 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.12.0.88
    Uninstalling opencv-python-headless-4.12.0.88:
      Successfully uninstalled opencv-python-headless-4.12.0.88
  Attempting uninstall: idna
    Found existing installation: idna 3.11
    Uninstalling idna-3.11:
      Successfully uninstalled idna-3.11
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.moun

#SAHI導入

In [ ]:
pip install ultralytics sahi

In [ ]:
import os
import glob
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction
import traceback

# ==========================================
# 1. 設定
# ==========================================
MODEL_PATH = "runs_yolo11/manganji_tracking/weights/best.pt"
INPUT_DIR = "/content/drive/MyDrive/1114yolov11/test/images"
OUTPUT_DIR = "inference_results_all"
CONFIDENCE_THRESHOLD = 0.25

# ==========================================
# 2. 実行処理
# ==========================================

def run_batch_inference():
    # 画像リスト取得
    image_paths = glob.glob(os.path.join(INPUT_DIR, "*.jpg"))
    image_paths += glob.glob(os.path.join(INPUT_DIR, "*.png"))

    if not image_paths:
        print(f"エラー: 画像が見つかりません。パス: {INPUT_DIR}")
        return

    print(f"対象枚数: {len(image_paths)} 枚")
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    print(f"--- モデル読み込み中... ---")

    detection_model = AutoDetectionModel.from_pretrained(
        model_type="yolov8",
        model_path=MODEL_PATH,
        confidence_threshold=CONFIDENCE_THRESHOLD,
        device="cuda",
    )
    print("モデル読み込み完了。推論を開始します。")

    # --- ループ処理 ---
    for i, img_path in enumerate(image_paths):
        file_name = os.path.basename(img_path)
        print(f"[{i+1}/{len(image_paths)}] 処理中: {file_name} ...", end="", flush=True)

        try:
            # 【修正】perform_standard_prediction 引数を削除しました
            result = get_sliced_prediction(
                img_path,
                detection_model,
                slice_height=640,
                slice_width=640,
                overlap_height_ratio=0.5, # 重なり率は維持（見逃し防止）
                overlap_width_ratio=0.5,
                verbose=0
            )

            file_name_no_ext = os.path.splitext(file_name)[0]
            result.export_visuals(export_dir=OUTPUT_DIR, file_name=file_name_no_ext)

            print(f" 完了 -> {len(result.object_prediction_list)}個検出")

        except Exception as e:
            print(f"\nエラーが発生しました ({file_name}): {e}")
            # エラーの詳細ログを表示（デバッグ用）
            # traceback.print_exc()

    print(f"\nすべての処理が完了しました。結果は '{OUTPUT_DIR}' を確認してください。")

if __name__ == "__main__":
    run_batch_inference()

対象枚数: 15 枚
--- モデル読み込み中... ---
モデル読み込み完了。推論を開始します。
[1/15] 処理中: GP__0046_JPG.rf.5305bd32e0df638c43922e4726ff08d7.jpg ... 完了 -> 62個検出
[2/15] 処理中: GP__0066_JPG.rf.c0cab49ff0b49d207c73942f5c68aefa.jpg ... 完了 -> 16個検出
[3/15] 処理中: GOPR0045_JPG.rf.8a8f48cd9941d3926eb022f430cd6a5f.jpg ... 完了 -> 24個検出
[4/15] 処理中: GOPR0027_JPG.rf.fd3a3e87014b5f5bf43ff1ea61544fad.jpg ... 完了 -> 59個検出
[5/15] 処理中: GOPR0013_JPG.rf.f4182d2da4070bac9cac7e9a854150f0.jpg ... 完了 -> 54個検出
[6/15] 処理中: GP__0073_JPG.rf.ccf2b63e0dd26e30b13236dbff997112.jpg ... 完了 -> 13個検出
[7/15] 処理中: GP__0018_JPG.rf.bc4fe248c17875b7e4c909e5189ec328.jpg ... 完了 -> 35個検出
[8/15] 処理中: GOPR0051_JPG.rf.0d08594045c26a90a447662f89d79710.jpg ... 完了 -> 64個検出
[9/15] 処理中: GOPR0069_JPG.rf.3e4725cb5263ea9c446ab3aabde7c55b.jpg ... 完了 -> 39個検出
[10/15] 処理中: GOPR0007_JPG.rf.73a46cf1cedc4a876c82d71550fb4fbf.jpg ... 完了 -> 30個検出
[11/15] 処理中: GP__0049_JPG.rf.e7b9932940ef24eff1b2448f7b41bf9c.jpg ... 完了 -> 37個検出
[12/15] 処理中: GP__0036_JPG.rf.5848d0a41bc341aea9f2f4f3a4c

In [ ]:
import shutil
from google.colab import files

# 1. フォルダ名と保存するZIPファイル名
folder_to_download = "inference_results_all"
zip_filename = "sahi_results.zip"

print(f"{folder_to_download} を圧縮しています...")

# 2. フォルダをZIP圧縮 (コマンドラインのzipを使用)
!zip -r {zip_filename} {folder_to_download}

print("圧縮完了。ダウンロードを開始します...")

# 3. ダウンロード実行
files.download(zip_filename)

inference_results_all を圧縮しています...
  adding: inference_results_all/ (stored 0%)
  adding: inference_results_all/GP__0073_JPG.rf.ccf2b63e0dd26e30b13236dbff997112.png (deflated 4%)
  adding: inference_results_all/GOPR0013_JPG.rf.f4182d2da4070bac9cac7e9a854150f0.png (deflated 3%)
  adding: inference_results_all/GOPR0037_JPG.rf.dd81e2d93e6f8a93bf81c86efcc97e2e.png (deflated 4%)
  adding: inference_results_all/GP__0049_JPG.rf.e7b9932940ef24eff1b2448f7b41bf9c.png (deflated 3%)
  adding: inference_results_all/GP__0082_JPG.rf.79e71085a14f02bff5d8545ae99815b2.png (deflated 5%)
  adding: inference_results_all/GOPR0051_JPG.rf.0d08594045c26a90a447662f89d79710.png (deflated 4%)
  adding: inference_results_all/GP__0066_JPG.rf.c0cab49ff0b49d207c73942f5c68aefa.png (deflated 4%)
  adding: inference_results_all/GP__0018_JPG.rf.bc4fe248c17875b7e4c909e5189ec328.png (deflated 4%)
  adding: inference_results_all/GOPR0045_JPG.rf.8a8f48cd9941d3926eb022f430cd6a5f.png (deflated 4%)
  adding: inference_results_al

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

##性能評価

In [ ]:
import json
import os
import glob
import cv2
import yaml
from datetime import datetime

# ==========================================
# 1. パス設定 (ここを書き換えてください)
# ==========================================

# data.yaml のパス (クラス名を取得するために読み込みます)
YAML_PATH = "/content/drive/MyDrive/1114yolov11/data.yaml"

# テスト用画像フォルダ
TEST_IMAGES_DIR = "/content/drive/MyDrive/1114yolov11/test/images"

# テスト用ラベル(YOLO形式txt)フォルダ
TEST_LABELS_DIR = "/content/drive/MyDrive/1114yolov11/test/labels"

# 出力するJSONファイルの名前
OUTPUT_JSON_PATH = "test_coco_format.json"

# ==========================================
# 2. 変換処理実行
# ==========================================

def yolo_to_coco():
    print("変換を開始します...")

    # data.yaml からクラス名を読み込む
    with open(YAML_PATH, 'r') as f:
        data_cfg = yaml.safe_load(f)
        class_names = data_cfg['names'] # リストまたは辞書

    # カテゴリ情報の作成
    categories = []
    # 辞書型の場合 {0: 'manganji', 1: 'leaf'} とリスト型 ['manganji', 'leaf'] の両対応
    if isinstance(class_names, dict):
        for k, v in class_names.items():
            categories.append({"id": int(k), "name": v})
    else:
        for i, name in enumerate(class_names):
            categories.append({"id": i, "name": name})

    images = []
    annotations = []
    ann_id = 0

    # 画像ファイルを取得
    img_files = glob.glob(os.path.join(TEST_IMAGES_DIR, "*.jpg"))
    img_files += glob.glob(os.path.join(TEST_IMAGES_DIR, "*.png"))

    if not img_files:
        print("エラー: 画像が見つかりません。パスを確認してください。")
        return

    for img_path in img_files:
        # 画像情報の取得 (高さ、幅が必要)
        img = cv2.imread(img_path)
        if img is None: continue
        h, w, _ = img.shape
        file_name = os.path.basename(img_path)
        image_id = file_name # ファイル名をIDとして使用

        images.append({
            "id": image_id,
            "file_name": file_name,
            "height": h,
            "width": w
        })

        # 対応するラベルファイルを探す
        label_file = os.path.splitext(file_name)[0] + ".txt"
        label_path = os.path.join(TEST_LABELS_DIR, label_file)

        if os.path.exists(label_path):
            with open(label_path, 'r') as f:
                lines = f.readlines()

            for line in lines:
                parts = line.strip().split()
                if len(parts) < 5: continue

                cls_id = int(parts[0])
                x_center = float(parts[1])
                y_center = float(parts[2])
                bbox_w = float(parts[3])
                bbox_h = float(parts[4])

                # YOLO形式(正規化座標) -> COCO形式(ピクセル座標 [x_min, y_min, w, h])
                abs_x = (x_center - bbox_w / 2) * w
                abs_y = (y_center - bbox_h / 2) * h
                abs_w = bbox_w * w
                abs_h = bbox_h * h

                annotations.append({
                    "id": ann_id,
                    "image_id": image_id,
                    "category_id": cls_id,
                    "bbox": [abs_x, abs_y, abs_w, abs_h],
                    "area": abs_w * abs_h,
                    "iscrowd": 0
                })
                ann_id += 1

    # JSON作成
    coco_format = {
        "images": images,
        "annotations": annotations,
        "categories": categories
    }

    with open(OUTPUT_JSON_PATH, 'w') as f:
        json.dump(coco_format, f)

    print(f"変換完了！ '{OUTPUT_JSON_PATH}' を作成しました。")
    print(f"画像数: {len(images)}, アノテーション数: {len(annotations)}")

if __name__ == "__main__":
    yolo_to_coco()

変換を開始します...
変換完了！ 'test_coco_format.json' を作成しました。
画像数: 15, アノテーション数: 259


In [ ]:
import json
import os
import glob
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

# ==========================================
# 設定
# ==========================================
MODEL_PATH = "runs_yolo11/manganji_tracking/weights/best.pt"
TEST_IMAGES_DIR = "/content/drive/MyDrive/1114yolov11/test/images"
COCO_GT_PATH = "test_coco_format.json" # さっき作った正解データ
RESULT_JSON_PATH = "results.json"      # これから作る推論結果

# SAHIパラメータ
SLICE_HEIGHT = 640
SLICE_WIDTH = 640
OVERLAP_RATIO = 0.5
CONFIDENCE_THRESHOLD = 0.25

# ==========================================
# 1. 準備: 正解データをロードして画像IDのマッピングを確認
# ==========================================
print("正解データを読み込んでいます...")
with open(COCO_GT_PATH, 'r') as f:
    gt_data = json.load(f)

# ファイル名から画像IDを引ける辞書を作る
# (Step1でIDをファイル名そのものにしている場合はそのまま使えますが、念のため)
filename_to_id = {img['file_name']: img['id'] for img in gt_data['images']}
print(f"評価対象: {len(filename_to_id)} 枚の画像")

# ==========================================
# 2. 推論実行 (SAHI loop)
# ==========================================
print("--- SAHIによる推論を開始します ---")

detection_model = AutoDetectionModel.from_pretrained(
    model_type="yolov8",
    model_path=MODEL_PATH,
    confidence_threshold=CONFIDENCE_THRESHOLD,
    device="cuda",
)

coco_results = [] # ここに結果を溜めていく

for file_name, image_id in filename_to_id.items():
    img_path = os.path.join(TEST_IMAGES_DIR, file_name)

    if not os.path.exists(img_path):
        continue

    # SAHI推論 (高精度設定)
    result = get_sliced_prediction(
        img_path,
        detection_model,
        slice_height=SLICE_HEIGHT,
        slice_width=SLICE_WIDTH,
        overlap_height_ratio=OVERLAP_RATIO,
        overlap_width_ratio=OVERLAP_RATIO,
        verbose=0
    )

    # 結果をCOCO形式に変換 [x_min, y_min, width, height]
    for obj in result.object_prediction_list:
        bbox = obj.bbox
        coco_results.append({
            "image_id": image_id,
            "category_id": obj.category.id,
            "bbox": [bbox.minx, bbox.miny, bbox.maxx - bbox.minx, bbox.maxy - bbox.miny],
            "score": obj.score.value
        })

print(f"推論完了。検出総数: {len(coco_results)}")

# 結果をJSONに保存
with open(RESULT_JSON_PATH, 'w') as f:
    json.dump(coco_results, f)

# ==========================================
# 3. 評価実行 (pycocotools)
# ==========================================
print("--- 精度評価 (mAP算出) ---")

# COCO APIを使って評価
cocoGt = COCO(COCO_GT_PATH)        # 正解データのロード
cocoDt = cocoGt.loadRes(RESULT_JSON_PATH) # 推論結果のロード

cocoEval = COCOeval(cocoGt, cocoDt, 'bbox')
cocoEval.evaluate()
cocoEval.accumulate()
cocoEval.summarize()

正解データを読み込んでいます...
評価対象: 15 枚の画像
--- SAHIによる推論を開始します ---
推論完了。検出総数: 604
--- 精度評価 (mAP算出) ---
loading annotations into memory...
Done (t=0.00s)
creating index...
index created!


KeyError: 'info'

In [ ]:
import json
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

# ==========================================
# 設定
# ==========================================
COCO_GT_PATH = "test_coco_format.json" # 正解データ
RESULT_JSON_PATH = "results.json"      # さっき作った推論結果

# ==========================================
# 1. JSONファイルの修正 (ここが修正ポイント)
# ==========================================
print("正解データのフォーマットを修正しています...")

with open(COCO_GT_PATH, 'r') as f:
    gt_data = json.load(f)

# 'info' キーがないとエラーになるため、ダミー情報を追加
if 'info' not in gt_data:
    gt_data['info'] = {
        "description": "Manganji Dataset",
        "url": "",
        "version": "1.0",
        "year": 2025,
        "contributor": "",
        "date_created": ""
    }

# 'licenses' キーも念のため追加
if 'licenses' not in gt_data:
    gt_data['licenses'] = []

# 修正したデータを上書き保存
with open(COCO_GT_PATH, 'w') as f:
    json.dump(gt_data, f)

print("修正完了。評価を続行します。")

# ==========================================
# 2. 評価実行 (pycocotools)
# ==========================================
print("\n--- 精度評価結果 (mAP) ---")

try:
    # COCO APIを使って評価
    cocoGt = COCO(COCO_GT_PATH)        # 正解データのロード
    cocoDt = cocoGt.loadRes(RESULT_JSON_PATH) # 推論結果のロード

    cocoEval = COCOeval(cocoGt, cocoDt, 'bbox')
    cocoEval.evaluate()
    cocoEval.accumulate()
    cocoEval.summarize()

except Exception as e:
    print(f"評価中にエラーが発生しました: {e}")

正解データのフォーマットを修正しています...
修正完了。評価を続行します。

--- 精度評価結果 (mAP) ---
loading annotations into memory...
Done (t=0.00s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.00s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=0.12s).
Accumulating evaluation results...
DONE (t=0.02s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.177
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.333
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.181
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.080
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.243
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.324
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.100
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.251
 Ave

#動画のフォーマット修正

In [ ]:
import os

# ===== 設定 =====
SRC_DIR = "/content/drive/MyDrive/testmovies"
OUT_DIR = "/content/drive/MyDrive/testmovies_fixed"
FPS = 30

os.makedirs(OUT_DIR, exist_ok=True)

# ===== 動画一括変換 =====
for fname in sorted(os.listdir(SRC_DIR)):
    if not fname.lower().endswith(".mp4"):
        continue

    src_path = os.path.join(SRC_DIR, fname)

    # 出力ファイル名（_fixed を付与）
    base, ext = os.path.splitext(fname)
    out_path = os.path.join(OUT_DIR, f"{base}_fixed{ext}")

    print(f"変換開始: {fname}")

    cmd = f'''
    ffmpeg -y -i "{src_path}" \
    -vf "scale=trunc(iw/2)*2:trunc(ih/2)*2,fps={FPS}" \
    -c:v libx264 -pix_fmt yuv420p -r {FPS} \
    -movflags +faststart -profile:v high -g {FPS} -bf 2 \
    "{out_path}"
    '''
    os.system(cmd)

    print(f"変換完了: {out_path}")

print("すべての動画変換が完了しました")


変換開始: testmovie1.1.MP4
変換完了: /content/drive/MyDrive/testmovies_fixed/testmovie1.1_fixed.MP4
変換開始: testmovie2.1.MP4
変換完了: /content/drive/MyDrive/testmovies_fixed/testmovie2.1_fixed.MP4
変換開始: testmovie2.2.MP4
変換完了: /content/drive/MyDrive/testmovies_fixed/testmovie2.2_fixed.MP4
変換開始: testmovie3.2.MP4
変換完了: /content/drive/MyDrive/testmovies_fixed/testmovie3.2_fixed.MP4
変換開始: testmoviw3.1.MP4
変換完了: /content/drive/MyDrive/testmovies_fixed/testmoviw3.1_fixed.MP4
すべての動画変換が完了しました


#RF-DETR + BoostTrack実装（これ1本で）

In [ ]:
!mkdir -p /content/drive/MyDrive/reid

In [ ]:
!wget -O /content/drive/MyDrive/reid/osnet_x0_25_msmt17.pt \
https://github.com/mikel-brostrom/yolo_tracking/releases/download/v0.1.0/osnet_x0_25_msmt17.pt

--2025-12-23 06:01:13--  https://github.com/mikel-brostrom/yolo_tracking/releases/download/v0.1.0/osnet_x0_25_msmt17.pt
Resolving github.com (github.com)... 20.205.243.166
Connecting to github.com (github.com)|20.205.243.166|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://github.com/mikel-brostrom/boxmot/releases/download/v0.1.0/osnet_x0_25_msmt17.pt [following]
--2025-12-23 06:01:13--  https://github.com/mikel-brostrom/boxmot/releases/download/v0.1.0/osnet_x0_25_msmt17.pt
Reusing existing connection to github.com:443.
HTTP request sent, awaiting response... 404 Not Found
2025-12-23 06:01:14 ERROR 404: Not Found.



In [ ]:
from pathlib import Path
print(Path("/content/drive/MyDrive/reid/osnet_x0_25_msmt17.pt").exists())

True


In [ ]:
# ============================================================
# RT-DETR (Ultralytics) + BoostTrack (BoxMOT) + Counting
# - Tracking: BoostTrack
# - Output: counted video + final counts print
# ============================================================

# 必要ならインストール（未実施の場合のみ）
# !pip install ultralytics boxmot opencv-python

import cv2
import numpy as np
import torch
from pathlib import Path

from ultralytics import RTDETR
from boxmot import BoostTrack

# ==========================================
# 1. 設定
# ==========================================
MODEL_PATH  = "/content/drive/MyDrive/manganji_rtdetr_best.pt"
VIDEO_PATH  = "/content/drive/MyDrive/testmovies_fixed/testmovie1.1_fixed.MP4"
OUTPUT_PATH = "/content/drive/MyDrive/testmovies_boosttrack_counted.mp4"

# クラス定義 (0: Dried, 1: Flower, 2: Fruit と仮定)
CLASS_NAMES = {0: "dried_flowers", 1: "flowers", 2: "fruits"}

# 色（BGR）
COLORS = {0: (92, 113, 166), 1: (220, 220, 255), 2: (0, 255, 0)}

CONF_THRESHOLD = 0.45

# ReIDモデル重み（必ず存在するパスにしてください）
REID_WEIGHTS = "/content/drive/MyDrive/reid/osnet_x0_25_msmt17.pt"

# ==========================================
# 2. 初期化
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

print(f"モデルロード中: {MODEL_PATH}")
model = RTDETR(MODEL_PATH)

reid_weights_path = Path(REID_WEIGHTS)
assert reid_weights_path.exists(), f"ReID重みが見つかりません: {reid_weights_path}"

tracker = BoostTrack(
    reid_weights=reid_weights_path,
    device=device,
    half=False,  # fp16にしたいなら True（挙動確認後に）
)

# カウント：クラスごとのユニークID集合
unique_id_sets = {cid: set() for cid in CLASS_NAMES.keys()}

cap = cv2.VideoCapture(VIDEO_PATH)
assert cap.isOpened(), f"動画を開けません: {VIDEO_PATH}"

w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS) or 30.0

writer = cv2.VideoWriter(
    OUTPUT_PATH,
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (w, h),
)

print(f"処理開始: {w}x{h} @ {fps}fps")

# ==========================================
# 3. メインループ
# ==========================================
frame_cnt = 0

with torch.inference_mode():
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        # --- RT-DETR 推論 ---
        results = model(frame, verbose=False)[0]

        # --- BoostTrackに渡す dets_array を作成 ---
        # 形式: (N,6) [x1,y1,x2,y2,conf,cls_id]
        dets_list = []
        for box in results.boxes:
            x1, y1, x2, y2 = box.xyxy[0].tolist()
            conf = float(box.conf[0])
            cls_id = int(box.cls[0])

            if conf >= CONF_THRESHOLD and cls_id in CLASS_NAMES:
                dets_list.append([x1, y1, x2, y2, conf, cls_id])

        dets_array = (
            np.array(dets_list, dtype=np.float32)
            if len(dets_list) > 0
            else np.empty((0, 6), dtype=np.float32)
        )

        # --- BoostTrack 更新 ---
        # 返り値（想定）: (M,8) [x1,y1,x2,y2,track_id,conf,cls_id,ind]
        tracks = tracker.update(dets_array, frame)
        tracks = np.asarray(tracks) if tracks is not None else np.empty((0, 8), dtype=np.float32)

        # --- 追跡・カウント・描画 ---
        if tracks.shape[0] > 0:
            for tr in tracks:
                x1, y1, x2, y2 = map(int, tr[0:4])
                track_id = int(tr[4])
                cls_id = int(tr[6])

                # カウント（クラス別ユニークID）
                if cls_id in unique_id_sets:
                    unique_id_sets[cls_id].add(track_id)

                color = COLORS.get(cls_id, (255, 255, 255))
                cls_name = CLASS_NAMES.get(cls_id, "unknown")

                cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
                cv2.putText(
                    frame,
                    f"{cls_name} id:{track_id}",
                    (x1, max(0, y1 - 10)),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.6,
                    color,
                    2,
                )

        # --- 画面へのカウント表示 ---
        overlay = frame.copy()
        cv2.rectangle(overlay, (5, 5), (320, 115), (0, 0, 0), -1)
        frame = cv2.addWeighted(overlay, 0.4, frame, 0.6, 0)

        y = 30
        cv2.putText(
            frame,
            "Counts (unique IDs):",
            (10, y),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (255, 255, 255),
            2,
        )

        for cid in sorted(CLASS_NAMES.keys()):
            y += 25
            txt = f"{CLASS_NAMES[cid]}: {len(unique_id_sets[cid])}"
            cv2.putText(
                frame,
                txt,
                (20, y),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.65,
                COLORS.get(cid, (255, 255, 255)),
                2,
            )

        writer.write(frame)

        frame_cnt += 1
        if frame_cnt % 50 == 0:
            print(
                f"processed: {frame_cnt} frames | "
                + ", ".join([f"{CLASS_NAMES[c]}={len(unique_id_sets[c])}" for c in sorted(CLASS_NAMES.keys())])
            )

cap.release()
writer.release()
print(f"完了！動画を保存しました: {OUTPUT_PATH}")

# ==========================================
# 4. 最終個数の出力（指定フォーマット）
# ==========================================
print("\n===== Final Counts =====")
for cid in sorted(CLASS_NAMES.keys()):
    print(f"{CLASS_NAMES[cid]}:{len(unique_id_sets[cid])}")
print("========================\n")


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


ModuleNotFoundError: No module named 'boxmot'

##一括処理ver


In [ ]:
# ============================================================
# Batch: RT-DETR (Ultralytics) + BoostTrack (BoxMOT) + Counting
# Input : /content/drive/MyDrive/testmovies_fixed/*.mp4(MP4)
# Output: /content/drive/MyDrive/testmovies_boosttrack_counted/{name}_counted.mp4
# Print : final counts per video
# ============================================================

# 必要ならインストール（未実施の場合のみ）
!pip install ultralytics boxmot opencv-python

import os
import cv2
import numpy as np
import torch
from pathlib import Path

from ultralytics import RTDETR
from boxmot import BoostTrack

# =========================
# 1) 設定
# =========================
MODEL_PATH = "/content/drive/MyDrive/manganji_rtdetr_best.pt"

SRC_DIR = "/content/drive/MyDrive/testmovies_fixed"
OUT_DIR = "/content/drive/MyDrive/testmovies_boosttrack_counted"
os.makedirs(OUT_DIR, exist_ok=True)

# クラス定義 (0: Dried, 1: Flower, 2: Fruit と仮定)
CLASS_NAMES = {0: "dried_flowers", 1: "flowers", 2: "fruits"}

# 色（BGR）
COLORS = {0: (92, 113, 166), 1: (220, 220, 255), 2: (0, 255, 0)}

#CONF_THRESHOLD = 0.45
#CONF_THRESHOLD = 0.60
CONF_THRESHOLD = 0.75
#CONF_THRESHOLD = 0.90

# ReIDモデル重み（必ず存在するパスにしてください）
REID_WEIGHTS = "/content/drive/MyDrive/reid/osnet_x0_25_msmt17.pt"

# 対象拡張子（mp4 / MP4）
VIDEO_EXTS = (".mp4", ".MP4")

# =========================
# 2) 初期化（モデルは一度だけ）
# =========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

print(f"モデルロード中: {MODEL_PATH}")
model = RTDETR(MODEL_PATH)

reid_weights_path = Path(REID_WEIGHTS)
assert reid_weights_path.exists(), f"ReID重みが見つかりません: {reid_weights_path}"

tracker = BoostTrack(
    reid_weights=reid_weights_path,
    device=device,
    half=False,  # fp16にしたいなら True（挙動確認後に）
)

# =========================
# 3) 入力動画リスト取得
# =========================
src_dir = Path(SRC_DIR)
assert src_dir.exists(), f"入力フォルダが存在しません: {SRC_DIR}"

video_paths = sorted([p for p in src_dir.iterdir() if p.suffix in VIDEO_EXTS or p.name.lower().endswith(".mp4")])
assert len(video_paths) > 0, f"動画が見つかりません: {SRC_DIR} (mp4/MP4)"

print(f"対象動画数: {len(video_paths)}")
for p in video_paths:
    print(" -", p.name)

# =========================
# 4) 動画ごとに処理
# =========================
for idx, video_path in enumerate(video_paths, start=1):
    # 動画ごとの出力
    out_path = Path(OUT_DIR) / f"{video_path.stem}_counted.mp4"

    # カウント：クラスごとのユニークID集合（動画ごとにリセット）
    unique_id_sets = {cid: set() for cid in CLASS_NAMES.keys()}

    cap = cv2.VideoCapture(str(video_path))
    assert cap.isOpened(), f"動画を開けません: {video_path}"

    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0

    writer = cv2.VideoWriter(
        str(out_path),
        cv2.VideoWriter_fourcc(*"mp4v"),
        fps,
        (w, h),
    )

    print("\n" + "="*60)
    print(f"[{idx}/{len(video_paths)}] 処理開始: {video_path.name} | {w}x{h} @ {fps:.2f}fps")
    print("="*60)

    frame_cnt = 0

    # ※重要：trackerの内部状態を動画ごとにクリアしたい場合がある
    # BoostTrack が reset を持っている環境なら呼ぶ（無ければそのまま続行）
    if hasattr(tracker, "reset") and callable(getattr(tracker, "reset")):
        tracker.reset()

    with torch.inference_mode():
        while True:
            ret, frame = cap.read()
            if not ret:
                break

            # --- RT-DETR 推論 ---
            results = model(frame, verbose=False)[0]

            # --- BoostTrackに渡す dets_array を作成 ---
            # 形式: (N,6) [x1,y1,x2,y2,conf,cls_id]
            dets_list = []
            for box in results.boxes:
                x1, y1, x2, y2 = box.xyxy[0].tolist()
                conf = float(box.conf[0])
                cls_id = int(box.cls[0])

                if conf >= CONF_THRESHOLD and cls_id in CLASS_NAMES:
                    dets_list.append([x1, y1, x2, y2, conf, cls_id])

            dets_array = (
                np.array(dets_list, dtype=np.float32)
                if len(dets_list) > 0
                else np.empty((0, 6), dtype=np.float32)
            )

            # --- BoostTrack 更新 ---
            # 返り値（想定）: (M,8) [x1,y1,x2,y2,track_id,conf,cls_id,ind]
            tracks = tracker.update(dets_array, frame)
            tracks = np.asarray(tracks) if tracks is not None else np.empty((0, 8), dtype=np.float32)

            # --- 追跡・カウント・描画 ---
            if tracks.shape[0] > 0:
                for tr in tracks:
                    x1, y1, x2, y2 = map(int, tr[0:4])
                    track_id = int(tr[4])
                    cls_id = int(tr[6])

                    # カウント（クラス別ユニークID）
                    if cls_id in unique_id_sets:
                        unique_id_sets[cls_id].add(track_id)

                    color = COLORS.get(cls_id, (255, 255, 255))
                    cls_name = CLASS_NAMES.get(cls_id, "unknown")

                    cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
                    cv2.putText(
                        frame,
                        f"{cls_name} id:{track_id}",
                        (x1, max(0, y1 - 10)),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        0.6,
                        color,
                        2,
                    )

            # --- 画面へのカウント表示 ---
            overlay = frame.copy()
            cv2.rectangle(overlay, (5, 5), (320, 115), (0, 0, 0), -1)
            frame = cv2.addWeighted(overlay, 0.4, frame, 0.6, 0)

            y = 30
            cv2.putText(
                frame,
                "Counts (unique IDs):",
                (10, y),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.7,
                (255, 255, 255),
                2,
            )

            for cid in sorted(CLASS_NAMES.keys()):
                y += 25
                txt = f"{CLASS_NAMES[cid]}: {len(unique_id_sets[cid])}"
                cv2.putText(
                    frame,
                    txt,
                    (20, y),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.65,
                    COLORS.get(cid, (255, 255, 255)),
                    2,
                )

            writer.write(frame)

            frame_cnt += 1
            if frame_cnt % 50 == 0:
                print(
                    f"processed: {frame_cnt} frames | "
                    + ", ".join([f"{CLASS_NAMES[c]}={len(unique_id_sets[c])}" for c in sorted(CLASS_NAMES.keys())])
                )

    cap.release()
    writer.release()

    print(f"完了！動画を保存しました: {out_path}")

    # --- 最終個数の出力（指定フォーマット） ---
    print("\n===== Final Counts =====")
    print(f"video:{video_path.name}")
    for cid in sorted(CLASS_NAMES.keys()):
        print(f"{CLASS_NAMES[cid]}:{len(unique_id_sets[cid])}")
    print("========================\n")

print("全動画の処理が完了しました。")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.0/178.0 kB 17.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 73.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 84.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 796.9/796.9 kB 70.1 MB/s eta 0:00:00
  Created wheel for filterpy: filename=filterpy-1.4.5-py3-none-any.whl size=110460 sha256=99197bdd79bb02039cf318ca6a2cb2b75e381b49708d1dcb8875165509b01352
  Stored in directory: /root/.cache/pip/wheels/77/bf/4c/b0c3f4798a0166668752312a67118b27a3cd341e13ac0ae6ee
Successfully built filterpy
  Attempting uninstall: regex
    Found existing installation: regex 2025.11.3
    Uninstalling regex-2025.11.3:
    

SUCCESS  | BoostTrack: det_thresh=0.3, max_age=30, max_obs=50, min_hits=3, iou_threshold=0.3, per_class=False, asso_func=iou, use_ecc=True, min_box_area=10, aspect_ratio_thresh=1.6, cmc_method=ecc, lambda_iou=0.5, lambda_mhd=0.25, lambda_shape=0.25, use_dlo_boost=True, use_duo_boost=True, dlo_boost_coef=0.65, s_sim_corr=False, use_rich_s=False, use_sb=False, use_vt=False, with_reid=False, reid=None


対象動画数: 5
 - testmovie1.1_fixed.MP4
 - testmovie2.1_fixed.MP4
 - testmovie2.2_fixed.MP4
 - testmovie3.2_fixed.MP4
 - testmoviw3.1_fixed.MP4

[1/5] 処理開始: testmovie1.1_fixed.MP4 | 1920x1080 @ 30.00fps
processed: 50 frames | dried_flowers=0, flowers=1, fruits=2
processed: 100 frames | dried_flowers=0, flowers=3, fruits=2
processed: 150 frames | dried_flowers=0, flowers=3, fruits=4
processed: 200 frames | dried_flowers=0, flowers=5, fruits=6
processed: 250 frames | dried_flowers=0, flowers=5, fruits=8
processed: 300 frames | dried_flowers=0, flowers=6, fruits=10
processed: 350 frames | dried_flowers=0, flowers=6, fruits=10
processed: 400 frames | dried_flowers=0, flowers=6, fruits=11
processed: 450 frames | dried_flowers=0, flowers=6, fruits=13
processed: 500 frames | dried_flowers=0, flowers=6, fruits=14
processed: 550 frames | dried_flowers=0, flowers=6, fruits=14
processed: 600 frames | dried_flowers=0, flowers=6, fruits=14
processed: 650 frames | dried_flowers=0, flowers=6, fruits=14
pr

WARNING  | ECC did not converge; returning identity warp.
WARNING  | ECC did not converge; returning identity warp.
WARNING  | ECC did not converge; returning identity warp.


processed: 900 frames | dried_flowers=2, flowers=5, fruits=9
processed: 950 frames | dried_flowers=2, flowers=5, fruits=9
processed: 1000 frames | dried_flowers=3, flowers=5, fruits=9
processed: 1050 frames | dried_flowers=3, flowers=5, fruits=10
processed: 1100 frames | dried_flowers=3, flowers=5, fruits=10
processed: 1150 frames | dried_flowers=3, flowers=6, fruits=12
processed: 1200 frames | dried_flowers=3, flowers=6, fruits=15
processed: 1250 frames | dried_flowers=3, flowers=6, fruits=15
processed: 1300 frames | dried_flowers=3, flowers=6, fruits=18
processed: 1350 frames | dried_flowers=3, flowers=7, fruits=19
processed: 1400 frames | dried_flowers=3, flowers=7, fruits=21
processed: 1450 frames | dried_flowers=3, flowers=8, fruits=22
processed: 1500 frames | dried_flowers=3, flowers=9, fruits=24
完了！動画を保存しました: /content/drive/MyDrive/testmovies_boosttrack_counted/testmoviw3.1_fixed_counted.mp4

===== Final Counts =====
video:testmoviw3.1_fixed.MP4
dried_flowers:3
flowers:9
fruits: